# PCA Example (Mall Customers Dataset)

Here it is demonstrated how to use the `PCA` module from the CMOR-438 library to reduce dimensionality and visualise structure.
In this example, the Mall Customers dataset is used to reveal which directions carry the most information.

**Goal: Understand the variance structure of the Mall Customers data and visualise K-Means clusters in principal component space.**

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os

# Add the algorithm folder to path
NOTEBOOK_DIR = os.path.abspath('')
sys.path.insert(0, NOTEBOOK_DIR)

# Data lives two levels up from the algorithm folder
DATA_DIR = os.path.join(NOTEBOOK_DIR, '..', '..', 'data')
# Also add the k_means folder for the cluster colouring
sys.path.insert(0, os.path.join(NOTEBOOK_DIR, '..', 'k_means_clustering'))

from pca import PCA
from k_means_clustering import KMeans
from sklearn.preprocessing import StandardScaler

mall = pd.read_csv(os.path.join(DATA_DIR, 'Mall_Customers.csv'))
MALL_FEATURES = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
X_raw = mall[MALL_FEATURES].values.astype(float)
X = StandardScaler().fit_transform(X_raw)

print(f"Dataset loaded: {mall.shape[0]} samples, {len(MALL_FEATURES)} features.")

## 2. Fit PCA

Fit PCA with 3 components (all features) and inspect explained variance.

In [ ]:
pca = PCA(n_components=3).fit(X)
X_pca = pca.transform(X)

print(f'Explained variance ratio: {pca.explained_variance_ratio_.round(4)}')
print(f'Total explained:          {pca.explained_variance_ratio_.sum()*100:.1f}%')
print(f'Reconstruction error:     {pca.reconstruction_error(X):.8f}')

## 3. K-Means Cluster Labels for Colouring

In [ ]:
km = KMeans(k=5, init='k-means++', n_init=5, random_state=42).fit(X)
labels_km = km.labels_
cmap = plt.cm.get_cmap('tab10', 5)

print(f'Cluster sizes: {dict(zip(*np.unique(labels_km, return_counts=True)))}')

## 4. Results and Visualisation

Scree plot, PC1 vs PC2 scatter coloured by cluster, and feature biplot.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(range(1,4), pca.explained_variance_ratio_, color=['steelblue','darkorange','seagreen'], edgecolor='white')
axes[0].plot(range(1,4), np.cumsum(pca.explained_variance_ratio_), 'o-', color='red', lw=1.5, label='Cumulative')
axes[0].set_xlabel('Principal Component'); axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('PCA - Scree Plot', fontweight='bold'); axes[0].legend()

for c in range(5):
    mask = labels_km==c
    axes[1].scatter(X_pca[mask,0], X_pca[mask,1], color=cmap(c), s=30, alpha=0.75, label=f'Cluster {c}')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
axes[1].set_title('PCA - PC1 vs PC2 (K-Means colours)', fontweight='bold'); axes[1].legend(fontsize=7)

loadings = pca.components_.T
for i, feat in enumerate(MALL_FEATURES):
    axes[2].arrow(0, 0, loadings[i,0], loadings[i,1], head_width=0.03, head_length=0.02, color=plt.cm.tab10(i), lw=2)
    axes[2].text(loadings[i,0]*1.12, loadings[i,1]*1.12, feat, fontsize=9, color=plt.cm.tab10(i), ha='center')
circle = plt.Circle((0,0),1,fill=False,color='gray',linestyle='--',lw=0.8)
axes[2].add_patch(circle)
axes[2].set_xlim(-1.3,1.3); axes[2].set_ylim(-1.3,1.3)
axes[2].set_xlabel('PC1 Loading'); axes[2].set_ylabel('PC2 Loading')
axes[2].set_title('PCA - Feature Biplot', fontweight='bold')
axes[2].axhline(0,color='gray',lw=0.5); axes[2].axvline(0,color='gray',lw=0.5)
plt.tight_layout(); plt.show()